In [1]:
import random
from pathlib import Path
from main import load_dotenv
from data_loader.raw_dataloader import RawDataloader
from models.ngram.knn import KNN
from models.ngram.model import Model
from collections import Counter, defaultdict

load_dotenv(Path("../.env"))

/home/bugslayer/Documents/research/MusicGeneration/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataloader = RawDataloader()
dataloader.prepare_dataset(representation="remi")
loader = dataloader.loader(load_music=dataloader.load_music)

In [3]:
ngram_model = Model(loader=loader, n=5)
ngram_model.remi_train()

In [4]:
ngram_model.models[0]

defaultdict(collections.Counter,
            {(-1, -1, -1, -1): Counter({4: 962}),
             (-1, -1, -1, 4): Counter({517: 962}),
             (-1, -1, 4, 517): Counter({190: 962}),
             (-1, 4, 517, 190): Counter({360: 962}),
             (4,
              517,
              190,
              360): Counter({206: 550,
                      205: 157,
                      207: 125,
                      204: 22,
                      203: 17,
                      200: 10,
                      208: 10,
                      199: 9,
                      202: 9,
                      201: 8,
                      197: 8,
                      196: 7,
                      191: 6,
                      192: 6,
                      195: 5,
                      198: 4,
                      193: 4,
                      194: 3,
                      380: 2}),
             (517, 190, 360, 207): Counter({380: 125}),
             (190,
              360,
              207,
    

In [7]:
next_token = None
tokens = (-1, -1,-1, -1)

while next_token != -1000:
    next_token = ngram_model.remi_predict(tokens)[0]
    tokens += (next_token,)

In [10]:
response = tokens[4:-1]

In [11]:
from data_processing.music_representations.decoders import DecodeContext, create_decoder
from data_processing.music_representations.helpers.adapters import canonical_frames_to_midi

# Generated tuples -> canonical -> MIDI. Nothing goes straight to MIDI: canonical is
# the only thing that writes a .mid, so this is the same path the built dataset takes.
decoder = create_decoder("remi", dataloader.config, dataloader.storage)
frames = decoder.decode(response, DecodeContext(piece_id="ngram_sample"))
frames["notes"].head()

piece_id,track_id,note_id,onset_tick,duration_tick,onset_quarter,duration_quarter,onset_sec,duration_sec,pitch,velocity
str,i16,i32,i64,i64,f64,f64,f64,f64,u8,u8
"""ngram_sample""",0,0,960,1020,2.0,2.125,0.989364,1.051199,64,55
"""ngram_sample""",0,1,1140,120,2.375,0.25,1.17487,0.12367,35,67
"""ngram_sample""",0,2,1440,720,3.0,1.5,1.484046,0.742023,54,71
"""ngram_sample""",0,3,1920,300,4.0,0.625,1.978728,0.309176,56,83
"""ngram_sample""",0,4,1920,300,4.0,0.625,1.978728,0.309176,68,79


In [12]:
midi_path = canonical_frames_to_midi(frames, Path("../generated/remi_sample.mid"))
print(midi_path, frames["notes"].height, "notes")


../generated/remi_sample.mid 2213 notes
